In [ ]:
!pip install -q pyspark

In [ ]:
import os
import zipfile
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive (1).zip to archive (1) (1).zip


In [ ]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

extract_path = "/content/movielens_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

print("\nFiles available:")
for file in os.listdir(extract_path):
    print(file)

Dataset extracted successfully!

Files available:
movies.csv
ratings.csv


In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 123285 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s

In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [ ]:
!java -version

openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32.1+1-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)


In [ ]:
import pyspark

print("PySpark version:", pyspark.__version__)

PySpark version: 4.0.4


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieLens 20M Analytics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("✅ Spark Session Started Successfully!")
print("Spark Version:", spark.version)

✅ Spark Session Started Successfully!
Spark Version: 4.0.4


In [ ]:
print("Spark Version:", spark.version)


Spark Version: 4.0.4


In [ ]:
movies_path = "/content/movielens_dataset/movies.csv"

movies_df = spark.read.csv(
    movies_path,
    header=True,
    inferSchema=True
)

print("Movies Dataset:")
movies_df.show(10, truncate=False)

print("Schema:")
movies_df.printSchema()

Movies Dataset:
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9   

In [ ]:
ratings_path = "/content/movielens_dataset/ratings.csv"

ratings_df = spark.read.csv(
    ratings_path,
    header=True,
    inferSchema=True
)

print("Ratings Dataset:")
ratings_df.show(10)

print("Schema:")
ratings_df.printSchema()

Ratings Dataset:
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|     16|   4.0|1217897793|
|     1|     24|   1.5|1217895807|
|     1|     32|   4.0|1217896246|
|     1|     47|   4.0|1217896556|
|     1|     50|   4.0|1217896523|
|     1|    110|   4.0|1217896150|
|     1|    150|   3.0|1217895940|
|     1|    161|   4.0|1217897864|
|     1|    165|   3.0|1217897135|
|     1|    204|   0.5|1217895786|
+------+-------+------+----------+
only showing top 10 rows
Schema:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [ ]:
print("Number of Movies:", movies_df.count())
print("Number of Ratings:", ratings_df.count())

Number of Movies: 10329
Number of Ratings: 105339


In [ ]:
print("Movies Columns:")
print(movies_df.columns)

print("\nRatings Columns:")
print(ratings_df.columns)

Movies Columns:
['movieId', 'title', 'genres']

Ratings Columns:
['userId', 'movieId', 'rating', 'timestamp']


In [ ]:
movies_df.show(10, truncate=False)
ratings_df.show(10)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9      |Sudden Death

In [ ]:
def missing_values(df, name):

    print(f"\n===== {name} =====")

    total = df.count()

    for column in df.columns:

        missing = df.filter(
            F.col(column).isNull()
        ).count()

        percentage = (missing / total) * 100

        print(
            f"{column}: {missing} missing "
            f"({percentage:.2f}%)"
        )

missing_values(movies_df, "Movies Dataset")
missing_values(ratings_df, "Ratings Dataset")


===== Movies Dataset =====
movieId: 0 missing (0.00%)
title: 0 missing (0.00%)
genres: 0 missing (0.00%)

===== Ratings Dataset =====
userId: 0 missing (0.00%)
movieId: 0 missing (0.00%)
rating: 0 missing (0.00%)
timestamp: 0 missing (0.00%)


In [ ]:
print("Duplicate Movies:")

movie_duplicates = movies_df.count() - \
                   movies_df.dropDuplicates().count()

print(movie_duplicates)


print("\nDuplicate Ratings:")

rating_duplicates = ratings_df.count() - \
                    ratings_df.dropDuplicates().count()

print(rating_duplicates)

Duplicate Movies:
0

Duplicate Ratings:
0


In [ ]:
invalid_ratings = ratings_df.filter(
    (F.col("rating") < 0.5) |
    (F.col("rating") > 5.0)
)

print("Invalid Ratings:", invalid_ratings.count())

invalid_ratings.show()

Invalid Ratings: 0
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
+------+-------+------+---------+



In [ ]:
clean_ratings_df = ratings_df \
    .dropna() \
    .filter(
        (F.col("rating") >= 0.5) &
        (F.col("rating") <= 5.0)
    ) \
    .dropDuplicates()

print("Original Ratings:", ratings_df.count())
print("Clean Ratings:", clean_ratings_df.count())

Original Ratings: 105339
Clean Ratings: 105339


In [ ]:
clean_movies_df = movies_df \
    .dropna(subset=["movieId", "title"]) \
    .withColumn(
        "title",
        F.trim(F.col("title"))
    ) \
    .dropDuplicates(["movieId"])

print("Original Movies:", movies_df.count())
print("Clean Movies:", clean_movies_df.count())

Original Movies: 10329
Clean Movies: 10329


In [ ]:
print("Current Ratings Partitions:",
      clean_ratings_df.rdd.getNumPartitions())

print("Current Movies Partitions:",
      clean_movies_df.rdd.getNumPartitions())

Current Ratings Partitions: 1
Current Movies Partitions: 1


In [ ]:
ratings_partitioned = clean_ratings_df.repartition(8)

print(
    "Ratings partitions after repartition:",
    ratings_partitioned.rdd.getNumPartitions()
)

Ratings partitions after repartition: 8


In [ ]:
movies_partitioned = clean_movies_df.repartition(4)

print(
    "Movies partitions after repartition:",
    movies_partitioned.rdd.getNumPartitions()
)

Movies partitions after repartition: 4


In [ ]:
ratings_partitioned.cache()
movies_partitioned.cache()

ratings_partitioned.count()
movies_partitioned.count()

print("✅ DataFrames cached successfully")

✅ DataFrames cached successfully


In [ ]:
rating_statistics = clean_ratings_df.select(
    F.min("rating").alias("minimum_rating"),
    F.max("rating").alias("maximum_rating"),
    F.avg("rating").alias("average_rating"),
    F.count("rating").alias("total_ratings")
)

rating_statistics.show()

+--------------+--------------+------------------+-------------+
|minimum_rating|maximum_rating|    average_rating|total_ratings|
+--------------+--------------+------------------+-------------+
|           0.5|           5.0|3.5168503593161127|       105339|
+--------------+--------------+------------------+-------------+



In [ ]:
rating_distribution = clean_ratings_df \
    .groupBy("rating") \
    .count() \
    .orderBy("rating")

rating_distribution.show()

+------+-----+
|rating|count|
+------+-----+
|   0.5| 1198|
|   1.0| 3258|
|   1.5| 1567|
|   2.0| 7943|
|   2.5| 5484|
|   3.0|21729|
|   3.5|12237|
|   4.0|28880|
|   4.5| 8187|
|   5.0|14856|
+------+-----+



In [ ]:
movie_rating_count = clean_ratings_df \
    .groupBy("movieId") \
    .agg(
        F.count("rating").alias("number_of_ratings")
    )

most_rated_movies = movie_rating_count \
    .join(
        clean_movies_df,
        "movieId",
        "inner"
    ) \
    .select(
        "movieId",
        "title",
        "number_of_ratings"
    ) \
    .orderBy(
        F.desc("number_of_ratings")
    )

most_rated_movies.show(20, truncate=False)

+-------+------------------------------------------------------------------------------+-----------------+
|movieId|title                                                                         |number_of_ratings|
+-------+------------------------------------------------------------------------------+-----------------+
|296    |Pulp Fiction (1994)                                                           |325              |
|356    |Forrest Gump (1994)                                                           |311              |
|318    |Shawshank Redemption, The (1994)                                              |308              |
|480    |Jurassic Park (1993)                                                          |294              |
|593    |Silence of the Lambs, The (1991)                                              |290              |
|260    |Star Wars: Episode IV - A New Hope (1977)                                     |273              |
|2571   |Matrix, The (1999)          

In [ ]:
movie_statistics = clean_ratings_df \
    .groupBy("movieId") \
    .agg(
        F.round(
            F.avg("rating"), 2
        ).alias("average_rating"),

        F.count("rating")
        .alias("number_of_ratings"),

        F.countDistinct("userId")
        .alias("distinct_users")
    )

movie_statistics.show(10)

+-------+--------------+-----------------+--------------+
|movieId|average_rating|number_of_ratings|distinct_users|
+-------+--------------+-----------------+--------------+
|   2947|          3.78|               58|            58|
|   5380|          3.64|                7|             7|
|  62956|          3.94|                8|             8|
|    348|          3.52|               28|            28|
|   1266|          3.92|               65|            65|
|   1104|          4.17|               32|            32|
|    783|          3.18|               54|            54|
|   1079|          3.95|              102|           102|
|   3101|          3.35|               50|            50|
|   7153|          4.11|              166|           166|
+-------+--------------+-----------------+--------------+
only showing top 10 rows


In [ ]:
producer_report = clean_movies_df \
    .join(
        movie_statistics,
        "movieId",
        "left"
    ) \
    .select(
        "movieId",
        "title",
        "average_rating",
        "number_of_ratings",
        "distinct_users"
    ) \
    .orderBy(
        F.desc("number_of_ratings")
    )

producer_report.show(
    20,
    truncate=False
)

+-------+------------------------------------------------------------------------------+--------------+-----------------+--------------+
|movieId|title                                                                         |average_rating|number_of_ratings|distinct_users|
+-------+------------------------------------------------------------------------------+--------------+-----------------+--------------+
|296    |Pulp Fiction (1994)                                                           |4.16          |325              |325           |
|356    |Forrest Gump (1994)                                                           |4.14          |311              |311           |
|318    |Shawshank Redemption, The (1994)                                              |4.45          |308              |308           |
|480    |Jurassic Park (1993)                                                          |3.66          |294              |294           |
|593    |Silence of the Lambs, The (1991)

use cases

In [ ]:
most_rated_movies.show(20, truncate=False)

+-------+------------------------------------------------------------------------------+-----------------+
|movieId|title                                                                         |number_of_ratings|
+-------+------------------------------------------------------------------------------+-----------------+
|296    |Pulp Fiction (1994)                                                           |325              |
|356    |Forrest Gump (1994)                                                           |311              |
|318    |Shawshank Redemption, The (1994)                                              |308              |
|480    |Jurassic Park (1993)                                                          |294              |
|593    |Silence of the Lambs, The (1991)                                              |290              |
|260    |Star Wars: Episode IV - A New Hope (1977)                                     |273              |
|2571   |Matrix, The (1999)          

In [ ]:
from pyspark.sql import functions as F

# Join ratings with movie information
svm_df = ratings_partitioned.join(
    movies_partitioned.select("movieId", "genres"),
    on="movieId",
    how="inner"
)

# Create binary target
# rating >= 4.0 → Liked (1)
# rating < 4.0  → Not Liked (0)

svm_df = svm_df.withColumn(
    "label",
    F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0)
)

svm_df.select(
    "userId",
    "movieId",
    "rating",
    "genres",
    "label"
).show(10, truncate=False)

+------+-------+------+-----------------------------+-----+
|userId|movieId|rating|genres                       |label|
+------+-------+------+-----------------------------+-----+
|113   |466    |2.5   |Action|Comedy|War            |0.0  |
|458   |3200   |4.0   |Comedy|Drama                 |1.0  |
|458   |4865   |4.0   |Crime|Horror|Mystery|Thriller|1.0  |
|54    |1438   |2.5   |Action|Thriller              |0.0  |
|402   |4015   |2.0   |Comedy|Sci-Fi                |0.0  |
|69    |8636   |3.5   |Action|Adventure|Sci-Fi|IMAX |0.0  |
|560   |1285   |4.5   |Comedy                       |1.0  |
|255   |4963   |4.0   |Crime|Thriller               |1.0  |
|133   |110    |4.0   |Action|Drama|War             |1.0  |
|605   |2852   |4.0   |Drama                        |1.0  |
+------+-------+------+-----------------------------+-----+
only showing top 10 rows


In [ ]:
from pyspark.ml.feature import RegexTokenizer

tokenizer = RegexTokenizer(
    inputCol="genres",
    outputCol="genre_words",
    pattern="\\|"
)

tokenized_df = tokenizer.transform(svm_df)

tokenized_df.select(
    "genres",
    "genre_words"
).show(10, truncate=False)

+-----------------------------+----------------------------------+
|genres                       |genre_words                       |
+-----------------------------+----------------------------------+
|Action|Comedy|War            |[action, comedy, war]             |
|Comedy|Drama                 |[comedy, drama]                   |
|Crime|Horror|Mystery|Thriller|[crime, horror, mystery, thriller]|
|Action|Thriller              |[action, thriller]                |
|Comedy|Sci-Fi                |[comedy, sci-fi]                  |
|Action|Adventure|Sci-Fi|IMAX |[action, adventure, sci-fi, imax] |
|Comedy                       |[comedy]                          |
|Crime|Thriller               |[crime, thriller]                 |
|Action|Drama|War             |[action, drama, war]              |
|Drama                        |[drama]                           |
+-----------------------------+----------------------------------+
only showing top 10 rows


In [ ]:
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(
    inputCol="genre_words",
    outputCol="features",
    binary=True
)

cv_model = cv.fit(tokenized_df)

featured_df = cv_model.transform(tokenized_df)

featured_df.select(
    "rating",
    "label",
    "features"
).show(10, truncate=False)

+------+-----+---------------------------------+
|rating|label|features                         |
+------+-----+---------------------------------+
|2.5   |0.0  |(20,[1,2,13],[1.0,1.0,1.0])      |
|4.0   |1.0  |(20,[0,1],[1.0,1.0])             |
|4.0   |1.0  |(20,[3,6,9,11],[1.0,1.0,1.0,1.0])|
|2.5   |0.0  |(20,[2,3],[1.0,1.0])             |
|2.0   |0.0  |(20,[1,7],[1.0,1.0])             |
|3.5   |0.0  |(20,[2,4,7,15],[1.0,1.0,1.0,1.0])|
|4.5   |1.0  |(20,[1],[1.0])                   |
|4.0   |1.0  |(20,[3,6],[1.0,1.0])             |
|4.0   |1.0  |(20,[0,2,13],[1.0,1.0,1.0])      |
|4.0   |1.0  |(20,[0],[1.0])                   |
+------+-----+---------------------------------+
only showing top 10 rows
